<a href="https://colab.research.google.com/github/bhavikd-ai/rag/blob/main/Hybrid_Search_in_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALLING REQUIRE PACKAGES

!pip install rank-bm25 sentence-transformers

In [3]:
# IMPORTING REQUIRE LIBRAIRES

import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# SAMPLE DOCUMENT

documents = [

    "RAG combines retrieval and generation for question answering.",
    "Vector databases store embeddings for semantic search.",
    "BM25 is a keyword based ranking algorithm.",
    "Reciprocal Rank Fusion combines multiple retrievers.",
    "LangChain helps build LLM applications.",
    "Dense retrieval uses embedding similarity.",
    "Sparse retrieval works well for exact keywords.",
    "FAISS is used for vector similarity search.",
    "Hybrid search combines BM25 and vector search."
]

In [5]:
# USER QUERY

query = "How to combine keyword and vector search in RAG?"

In [7]:
# BM25 PARSE RETRIEVAL

# Tokenize documents
tokenized_docs = [doc.lower().split() for doc in documents]

# Create BM25 model
bm25 = BM25Okapi(tokenized_docs)

# Tokenize query
tokenized_query = query.lower().split()

# BM25 scores
bm25_scores = bm25.get_scores(tokenized_query)

# Print BM25 results
for idx, score in enumerate(bm25_scores):

    print(f"Document : {documents[idx]}")
    print(f"BM25 Score: {score:.4f}")
    print()

Document : RAG combines retrieval and generation for question answering.
BM25 Score: 0.9995

Document : Vector databases store embeddings for semantic search.
BM25 Score: 0.6007

Document : BM25 is a keyword based ranking algorithm.
BM25 Score: 1.6832

Document : Reciprocal Rank Fusion combines multiple retrievers.
BM25 Score: 0.0000

Document : LangChain helps build LLM applications.
BM25 Score: 0.0000

Document : Dense retrieval uses embedding similarity.
BM25 Score: 0.0000

Document : Sparse retrieval works well for exact keywords.
BM25 Score: 0.0000

Document : FAISS is used for vector similarity search.
BM25 Score: 0.6007

Document : Hybrid search combines BM25 and vector search.
BM25 Score: 3.3500



In [ ]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")


In [16]:
# DENSE RETRIEVER

# Create embeddings
doc_embeddings = model.encode(documents)

query_embedding = model.encode([query])

# Cosine similarity
dense_scores = cosine_similarity(
    query_embedding,
    doc_embeddings
)[0]

# Print dense retrieval results
for idx, score in enumerate(dense_scores):

    print(f"Document : {documents[idx]}")
    print(f"Dense Score: {score:.4f}")
    print()

Document : RAG combines retrieval and generation for question answering.
Dense Score: 0.5088

Document : Vector databases store embeddings for semantic search.
Dense Score: 0.4191

Document : BM25 is a keyword based ranking algorithm.
Dense Score: 0.2379

Document : Reciprocal Rank Fusion combines multiple retrievers.
Dense Score: 0.2249

Document : LangChain helps build LLM applications.
Dense Score: 0.0870

Document : Dense retrieval uses embedding similarity.
Dense Score: 0.3177

Document : Sparse retrieval works well for exact keywords.
Dense Score: 0.4401

Document : FAISS is used for vector similarity search.
Dense Score: 0.3942

Document : Hybrid search combines BM25 and vector search.
Dense Score: 0.5211



In [15]:
bm25_weight = 0.4
dense_weight = 0.6
alpha = 0.6

df = pd.DataFrame()
df["documents"] = documents
df["bm25_scores"] = bm25_scores
df["dense_scores"] = dense_scores
df["bm25_normalized"] = ((df["bm25_scores"] - df["bm25_scores"].min()) / (df["bm25_scores"].max() - df["bm25_scores"].min()))
df["dense_normalized"] = ((df["dense_scores"] - df["dense_scores"].min()) / (df["dense_scores"].max() - df["dense_scores"].min()))
df["hybrid_score"] = (bm25_weight * df["bm25_normalized"] + dense_weight * df["dense_normalized"])
# df["hybrid_score2"] = (alpha * df["dense_normalized"] + (1 - alpha) * df["bm25_normalized"])
df = df.sort_values(by="hybrid_score", ascending=False)

df

,documents,bm25_scores,dense_scores,bm25_normalized,dense_normalized,hybrid_score,hybrid_score2
8,Hybrid search combines BM25 and vector search.,3.350048,0.521124,1.000000,1.000000,1.000000,1.000000
0,RAG combines retrieval and generation for ques...,0.999508,0.508803,0.298356,0.971620,0.702314,0.702314
1,Vector databases store embeddings for semantic...,0.600712,0.419115,0.179315,0.765032,0.530745,0.530745
7,FAISS is used for vector similarity search.,0.600712,0.394250,0.179315,0.707757,0.496380,0.496380
6,Sparse retrieval works well for exact keywords.,0.000000,0.440105,0.000000,0.813381,0.488028,0.488028
2,BM25 is a keyword based ranking algorithm.,1.683248,0.237874,0.502455,0.347558,0.409517,0.409517
5,Dense retrieval uses embedding similarity.,0.000000,0.317706,0.000000,0.531444,0.318866,0.318866
3,Reciprocal Rank Fusion combines multiple retri...,0.000000,0.224947,0.000000,0.317783,0.190670,0.190670
4,LangChain helps build LLM applications.,0.000000,0.086986,0.000000,0.000000,0.000000,0.000000


In [2]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Hybrid_Search_in_RAG.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove widget metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Cleaned successfully")

Cleaned successfully
